In [ ]:
import pandas as pd
import joblib # Dùng để lưu mô hình
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# 1. Đọc dữ liệu đã xử lý (Thay đổi đường dẫn file của nhóm bạn)
df = pd.read_csv('data_da_xu_ly.csv')
X = df.drop('stroke', axis=1)
y = df['stroke']

# 5.1 Chia dữ liệu 80/20 với Stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Danh sách lưu kết quả tốt nhất
best_models = {}

# 5.2 & 5.3 Huấn luyện và Tối ưu (GridSearchCV)

# A. Decision Tree
param_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}
grid_dt = GridSearchCV(DecisionTreeClassifier(), param_dt, cv=5, n_jobs=-1)
grid_dt.fit(X_train, y_train)
best_models['DecisionTree'] = grid_dt.best_estimator_

# B. Naive Bayes (Thường ít tham số để tối ưu bằng Grid)
nb = GaussianNB()
nb.fit(X_train, y_train)
best_models['NaiveBayes'] = nb

# C. SVM
param_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}
grid_svm = GridSearchCV(
    SVC(probability=True),
    param_svm,
    cv=5,
    n_jobs=-1
)
grid_svm.fit(X_train, y_train)
best_models['SVM'] = grid_svm.best_estimator_

# D. Neural Network (MLP)
param_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'activation': ['tanh', 'relu'],
    'solver': ['adam', 'sgd'],
    'max_iter': [500] 
}
grid_mlp = GridSearchCV(MLPClassifier(), param_mlp, cv=5, n_jobs=-1)
grid_mlp.fit(X_train, y_train)
best_models['NeuralNetwork'] = grid_mlp.best_estimator_

In [ ]:
print("--- KẾT QUẢ TỐI ƯU ---")
for name, model in best_models.items():
    # In thông số tốt nhất ra màn hình để đưa vào báo cáo
    if name != 'NaiveBayes':
        print(f"{name} - Best Params: {model.get_params()}")
    
    # Lưu mô hình thành file .pkl để sử dụng sau này
    joblib.dump(model, f'model_{name}.pkl')
    
print("\nĐã lưu tất cả mô hình và thông số thành công!")